# Align box to headset

Example of how to generate simulation transformations that align the box based on the XR headset.

<video controls src="../figures/align-box.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")
simulation.load()

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: box alignment")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Commands

In [ ]:
BOX_SCALE = .25
BOX_OFFSET = 1

In [48]:
import numpy as np
from nanover.utilities.transforms import Transform

def make_scaled_box_origin_transform(*, scale=1.0):
    return Transform.from_stack(
        Transform.x_axis_flip(),
        Transform.from_trs(scale=[scale, scale, scale]),
        Transform.from_trs(translation=utilities.box_size * -.5),
    )

def make_transform_box_scaled_and_in_front_of_headset(*, scale=1.0, distance=1.0):
    box = make_scaled_box_origin_transform(scale=scale)
    avatar = get_first_avatar_transform(snap_rotation=True)
    forward = Transform.from_trs(translation=(0, 0, distance))

    return Transform.from_stack(avatar, forward, box)

def snap_quaternion_to_y_axis(quaternion):
    snapped = np.array([0, quaternion[1], 0, quaternion[3]])
    snapped /= np.linalg.norm(snapped)
    return snapped

def get_first_avatar_position_rotation():
    avatar = next((value for key, value in utilities.copy_shared_state().items() if key.startswith("avatar.")))
    head = next((value for value in avatar["components"] if "headset" in value["name"]))

    return head["position"], head["rotation"]

def get_first_avatar_transform(snap_rotation=False):
    position, rotation = get_first_avatar_position_rotation()

    if snap_rotation:
        rotation = snap_quaternion_to_y_axis(rotation)

    transform = Transform.from_state_transform([*position, *rotation])
    return transform

def box_identity():
    utilities.scene_transform = Transform.x_axis_flip()

def box_origin():
    utilities.scene_transform = make_scaled_box_origin_transform(scale=BOX_SCALE)

def box_head():
    utilities.scene_transform = make_transform_box_scaled_and_in_front_of_headset(scale=BOX_SCALE, distance=BOX_OFFSET)


utilities.define_command("user/box/identity", handler=box_identity, label="identity", icon="🧭")
utilities.define_command("user/box/origin", handler=box_origin, label="origin", icon="⏺️")
utilities.define_command("user/box/head", handler=box_head, label="ahead", icon="⬆️")